# YouTube Channel Manager (Colab)

Interactive downloader with **per-channel filtering**. All settings persist to Drive.

**Folder structure**
```
/jianglens/
  youtube/
    _config.json           ← channel list + filter rules (auto-saved)
    <input_handle>/
      _channel.json
      <video_id>/audio.wav
    ...
```

**Flow**
1. Run **Setup** — mount Drive, install deps, load config
2. **Channel Manager** — add / remove YouTube channels
3. **Browse & Filter** — view videos, set per-channel filter rules
4. **Sync** — download filtered videos as WAV to Drive

In [ ]:
#@title 0) Setup
#@markdown Mount Drive · install deps · load saved config

import os, json, re, subprocess, shutil, time
from pathlib import Path

# ── Colab detection & Drive mount ──────────────────────────
def _is_colab():
    try:
        import google.colab
        return True
    except Exception:
        return False

IS_COLAB = _is_colab()
if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

# ── Paths ──────────────────────────────────────────────────
LENS_COMPRESSION_DRIVE_ROOT = "/content/drive/MyDrive/jianglens"
LENS_COMPRESSION_YOUTUBE_ROOT = LENS_COMPRESSION_DRIVE_ROOT + "/youtube"
CONFIG_PATH = LENS_COMPRESSION_YOUTUBE_ROOT + "/_config.json"
COOKIES_PATH = LENS_COMPRESSION_DRIVE_ROOT + "/cookies.txt"
AUDIO_QUALITY = "192"

Path(LENS_COMPRESSION_YOUTUBE_ROOT).mkdir(parents=True, exist_ok=True)

# ── Install dependencies ──────────────────────────────────
subprocess.run(["pip", "install", "-q", "yt-dlp"], check=True)
if not shutil.which("deno"):
    subprocess.run(["npm", "install", "-g", "deno"], check=True)
if not shutil.which("ffmpeg"):
    subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"], check=True)

import yt_dlp
import ipywidgets as widgets
from IPython.display import display, clear_output

# ── Config persistence (JSON on Drive) ─────────────────────
def load_config():
    if Path(CONFIG_PATH).exists():
        return json.loads(Path(CONFIG_PATH).read_text())
    return {"channels": []}

def save_config(cfg):
    Path(CONFIG_PATH).write_text(json.dumps(cfg, indent=2))

config = load_config()

# ── Cookies ────────────────────────────────────────────────
if COOKIES_PATH and not Path(COOKIES_PATH).exists():
    print(f"\u26a0\ufe0f  cookies.txt not found at {COOKIES_PATH}")
    COOKIES_PATH = ""

# ── yt-dlp helpers ─────────────────────────────────────────
_COOKIE_ERROR_PATTERNS = [
    "sign in", "login required", "cookies", "age-restricted",
    "private video", "confirm your age", "bot detection", "captcha",
]

class CookieExpiredError(Exception):
    pass

def _check_cookie_error(msg):
    low = msg.lower()
    for p in _COOKIE_ERROR_PATTERNS:
        if p in low:
            raise CookieExpiredError(
                f"Cookie/auth failure: {msg[:300]}\n"
                f"Re-export cookies.txt \u2192 {COOKIES_PATH or 'Drive'}")

def _base_opts():
    opts = {
        "quiet": True, "no_warnings": True,
        "js_runtimes": {"deno": {}},
        "remote_components": ["ejs:github"],
    }
    if COOKIES_PATH:
        opts["cookiefile"] = COOKIES_PATH
    return opts

def _channel_url(ch):
    ch = ch.strip()
    if ch.startswith("http"): return ch.rstrip("/")
    if ch.startswith("@"): return f"https://www.youtube.com/{ch}"
    if ch.startswith("UC"): return f"https://www.youtube.com/channel/{ch}"
    return f"https://www.youtube.com/{ch}"

def _fetch_tab(url):
    """Fetch a single channel tab, return (channel_id, [video_dicts])."""
    opts = _base_opts()
    opts["extract_flat"] = True
    opts["extractor_args"] = {"youtubetab": {"approximate_date": [""]}}
    channel_id, videos = None, []
    with yt_dlp.YoutubeDL(opts) as ydl:
        info = ydl.extract_info(url, download=False)
        channel_id = info.get("channel_id") or info.get("uploader_id")
        for e in (info.get("entries") or []):
            if not e: continue
            v = {
                "id": e.get("id"),
                "title": e.get("title", ""),
                "duration": e.get("duration") or 0,
                "upload_date": e.get("upload_date", ""),
            }
            if v["id"]:
                if not channel_id or not channel_id.startswith("UC"):
                    channel_id = e.get("channel_id") or channel_id
                videos.append(v)
    return channel_id, videos

def fetch_channel_videos(channel):
    """Fetch both /videos and /streams tabs, merge and deduplicate."""
    base = _channel_url(channel)
    channel_id, all_videos = None, {}
    try:
        for tab in ("/videos", "/streams"):
            try:
                cid, vids = _fetch_tab(base + tab)
                if cid:
                    channel_id = cid
                for v in vids:
                    all_videos[v["id"]] = v  # dedup by id
            except Exception as e:
                # If /streams tab doesn't exist, skip it
                if tab == "/streams":
                    continue
                raise
    except Exception as e:
        _check_cookie_error(str(e))
        raise
    if not channel_id:
        channel_id = channel
    videos = list(all_videos.values())
    videos.sort(key=lambda v: v.get("upload_date", ""), reverse=True)
    return channel_id, videos

def apply_filters(videos, filters):
    # Split videos into (passing, filtered_out, manually_excluded).
    min_dur = filters.get("min_duration", 0)
    max_dur = filters.get("max_duration", 0)
    title_inc = filters.get("title_include", "")
    title_exc = filters.get("title_exclude", "")
    exclude_ids = set(filters.get("exclude_ids", []))

    passing, filtered, excluded = [], [], []
    for v in videos:
        if v["id"] in exclude_ids:
            excluded.append(v)
            continue
        skip = False
        if min_dur > 0 and v["duration"] < min_dur:
            skip = True
        if max_dur > 0 and v["duration"] > max_dur:
            skip = True
        if not skip and title_inc:
            try:
                if not re.search(title_inc, v["title"], re.IGNORECASE):
                    skip = True
            except re.error:
                pass
        if not skip and title_exc:
            try:
                if re.search(title_exc, v["title"], re.IGNORECASE):
                    skip = True
            except re.error:
                pass
        (filtered if skip else passing).append(v)
    return passing, filtered, excluded

def get_downloaded_ids(channel_dir):
    ids = set()
    if channel_dir.exists():
        for d in channel_dir.iterdir():
            if d.is_dir() and ((d / "audio.wav").exists() or (d / "video.wav").exists()):
                ids.add(d.name)
    return ids

def download_to_tmp(video_id):
    tmp_path = f"/tmp/{video_id}.%(ext)s"
    wav_path = f"/tmp/{video_id}.wav"
    if os.path.exists(wav_path):
        os.remove(wav_path)
    opts = _base_opts()
    opts.update({
        "format": "bestaudio/best",
        "postprocessors": [{
            "key": "FFmpegExtractAudio",
            "preferredcodec": "wav",
            "preferredquality": AUDIO_QUALITY,
        }],
        "outtmpl": tmp_path,
        "keepvideo": False,
        "noplaylist": True,
    })
    try:
        with yt_dlp.YoutubeDL(opts) as ydl:
            ydl.download([f"https://www.youtube.com/watch?v={video_id}"])
    except Exception as e:
        _check_cookie_error(str(e))
        print(f"      \u274c {str(e)[:200]}")
        return None
    return wav_path if os.path.exists(wav_path) else None

def copy_to_drive(wav_path, dest_dir):
    dest_dir.mkdir(parents=True, exist_ok=True)
    shutil.move(wav_path, str(dest_dir / "audio.wav"))

def fmt_dur(s):
    s = int(s)
    if s >= 3600:
        return f"{s//3600}h{(s%3600)//60:02d}m"
    return f"{s//60}m{s%60:02d}s"

# ── In-memory video cache (avoids re-fetching during sync) ─
_video_cache = {}

print("\u2705 Setup complete")
print(f"   Channels: {len(config.get('channels', []))} configured")
print(f"   Cookies:  {'enabled' if COOKIES_PATH else 'disabled'}")
print(f"   Config:   {CONFIG_PATH}")

In [ ]:
#@title 1) Channel Manager
#@markdown Add or remove YouTube channels. Changes auto-save to Drive.

out_list = widgets.Output()
out_status = widgets.Output()

def _render_channels():
    with out_list:
        clear_output()
        chs = config.get("channels", [])
        if not chs:
            print("  No channels yet \u2014 add one below.")
            return
        for i, ch in enumerate(chs):
            f = ch.get("filters", {})
            tags = []
            if f.get("min_duration", 0) > 0:
                tags.append(f"\u2265{f['min_duration']//60}min")
            if f.get("max_duration", 0) > 0:
                tags.append(f"\u2264{f['max_duration']//60}min")
            if f.get("title_include"):
                tags.append(f"has '{f['title_include']}'")
            if f.get("title_exclude"):
                tags.append(f"skip '{f['title_exclude']}'")
            n = len(f.get("exclude_ids", []))
            if n:
                tags.append(f"{n} excluded")
            fstr = f"  [{', '.join(tags)}]" if tags else ""
            print(f"  {i+1}. {ch['input']}  \u2192  {ch.get('channel_id', '?')}{fstr}")

# ── Add channel ────────────────────────────────────────────
txt_add = widgets.Text(
    placeholder="@handle, URL, or UCxxxx",
    layout=widgets.Layout(width="350px"),
)
btn_add = widgets.Button(description="Add", button_style="primary", icon="plus")

def _on_add(_):
    val = txt_add.value.strip()
    if not val:
        return
    for ch in config.get("channels", []):
        if ch["input"] == val or ch.get("channel_id") == val:
            with out_status:
                clear_output()
                print(f"\u26a0\ufe0f Already exists: {val}")
            return
    with out_status:
        clear_output()
        print(f"Resolving {val}...")
    try:
        cid, vids = fetch_channel_videos(val)
    except Exception as e:
        with out_status:
            clear_output()
            print(f"\u274c {e}")
        return
    config.setdefault("channels", []).append({
        "input": val,
        "channel_id": cid,
        "filters": {
            "min_duration": 0,
            "max_duration": 0,
            "title_include": "",
            "title_exclude": "",
            "exclude_ids": [],
        },
    })
    save_config(config)
    cdir = Path(LENS_COMPRESSION_YOUTUBE_ROOT) / val
    cdir.mkdir(parents=True, exist_ok=True)
    mp = cdir / "_channel.json"
    if not mp.exists():
        mp.write_text(json.dumps({
            "input": val, "channel_id": cid,
            "added": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, indent=2))
    txt_add.value = ""
    with out_status:
        clear_output()
        print(f"\u2705 Added {val} \u2192 {cid} ({len(vids)} videos)")
    _render_channels()
    _refresh_dd()

btn_add.on_click(_on_add)

# ── Remove channel ─────────────────────────────────────────
dd_rm = widgets.Dropdown(
    options=[],
    description="",
    layout=widgets.Layout(width="350px"),
)
btn_rm = widgets.Button(description="Remove", button_style="danger", icon="trash")

def _refresh_dd():
    chs = config.get("channels", [])
    dd_rm.options = (
        [(f"{ch['input']}", i) for i, ch in enumerate(chs)]
        if chs else []
    )

def _on_rm(_):
    if dd_rm.value is None or not config.get("channels"):
        return
    removed = config["channels"].pop(dd_rm.value)
    save_config(config)
    with out_status:
        clear_output()
        print(f"Removed: {removed['input']}")
    _render_channels()
    _refresh_dd()

btn_rm.on_click(_on_rm)
_refresh_dd()

# ── Layout ─────────────────────────────────────────────────
display(widgets.HTML("<h3>\U0001f4fa Channel Manager</h3>"))
display(out_list)
display(widgets.HTML("<b>Add channel</b>"))
display(widgets.HBox([txt_add, btn_add]))
display(widgets.HTML("<b>Remove channel</b>"))
display(widgets.HBox([dd_rm, btn_rm]))
display(out_status)
_render_channels()

In [ ]:
#@title 2) Browse & Filter
#@markdown Select a channel, set filter rules, preview which videos pass.
#@markdown Filters are saved to Drive and persist across sessions.

_chs = config.get("channels", [])

if not _chs:
    print("\u26a0\ufe0f No channels. Add one in the Channel Manager, then re-run this cell.")
else:
    # ── Widgets ────────────────────────────────────────────
    dd_ch = widgets.Dropdown(
        options=[(ch["input"], i) for i, ch in enumerate(_chs)],
        description="Channel:",
    )
    int_min = widgets.IntText(
        value=0, description="Min (min):",
        layout=widgets.Layout(width="180px"),
    )
    int_max = widgets.IntText(
        value=0, description="Max (min):",
        layout=widgets.Layout(width="180px"),
    )
    txt_inc = widgets.Text(
        value="", description="Title has:",
        placeholder="regex", layout=widgets.Layout(width="280px"),
    )
    txt_exc = widgets.Text(
        value="", description="Title !has:",
        placeholder="regex", layout=widgets.Layout(width="280px"),
    )
    txt_ids = widgets.Textarea(
        value="",
        placeholder="paste video IDs to exclude, one per line",
        layout=widgets.Layout(width="450px", height="60px"),
    )
    btn_save = widgets.Button(
        description="Save Filters", button_style="success", icon="save",
    )
    btn_fetch = widgets.Button(
        description="Fetch & Preview", button_style="primary", icon="search",
    )
    out_pv = widgets.Output()

    # ── Load filters when channel changes ──────────────────
    def _load_f(change=None):
        idx = dd_ch.value
        if idx is None:
            return
        f = config["channels"][idx].get("filters", {})
        int_min.value = f.get("min_duration", 0) // 60
        int_max.value = f.get("max_duration", 0) // 60
        txt_inc.value = f.get("title_include", "")
        txt_exc.value = f.get("title_exclude", "")
        txt_ids.value = "\n".join(f.get("exclude_ids", []))

    dd_ch.observe(_load_f, names="value")
    _load_f()

    # ── Save filters to config ─────────────────────────────
    def _save_f():
        idx = dd_ch.value
        config["channels"][idx]["filters"] = {
            "min_duration": int_min.value * 60,
            "max_duration": int_max.value * 60,
            "title_include": txt_inc.value.strip(),
            "title_exclude": txt_exc.value.strip(),
            "exclude_ids": [
                x.strip()
                for x in txt_ids.value.strip().split("\n")
                if x.strip()
            ],
        }
        save_config(config)

    def _on_save(_):
        _save_f()
        with out_pv:
            clear_output()
            print("\u2705 Filters saved to Drive!")

    def _on_fetch(_):
        _save_f()  # auto-save current values before preview
        idx = dd_ch.value
        ch = config["channels"][idx]
        inp = ch["input"]
        with out_pv:
            clear_output()
            print(f"Fetching {inp}...")

        try:
            cid, videos = fetch_channel_videos(inp)
        except Exception as e:
            with out_pv:
                clear_output()
                print(f"\u274c {e}")
            return

        # Update channel_id if needed
        if cid and cid != ch.get("channel_id"):
            config["channels"][idx]["channel_id"] = cid
            save_config(config)

        _video_cache[inp] = videos

        filters = config["channels"][idx]["filters"]
        passing, filtered_out, excluded = apply_filters(videos, filters)
        already = get_downloaded_ids(Path(LENS_COMPRESSION_YOUTUBE_ROOT) / inp)
        new = [v for v in passing if v["id"] not in already]
        done = [v for v in passing if v["id"] in already]

        with out_pv:
            clear_output()
            print(f"\U0001f4ca {inp} ({cid})")
            print(
                f"   Total: {len(videos)} \u2502 "
                f"Pass filter: {len(passing)} \u2502 "
                f"Filtered: {len(filtered_out)} \u2502 "
                f"Excluded: {len(excluded)}"
            )
            print(
                f"   Downloaded: {len(done)} \u2502 "
                f"New to download: {len(new)}"
            )

            if new:
                print(f"\n\U0001f195 Will download ({len(new)}):")
                for v in new:
                    print(
                        f"   {v['id']}  {fmt_dur(v['duration']):>8s}  "
                        f"{v.get('upload_date',''):10s}  {v['title'][:55]}"
                    )

            if filtered_out:
                print(f"\n\U0001f53d Filtered out ({len(filtered_out)}):")
                for v in filtered_out[:20]:
                    print(
                        f"   {v['id']}  {fmt_dur(v['duration']):>8s}  "
                        f"{v.get('upload_date',''):10s}  {v['title'][:55]}"
                    )
                if len(filtered_out) > 20:
                    print(f"   ... +{len(filtered_out) - 20} more")

            if excluded:
                print(f"\n\U0001f6ab Manually excluded ({len(excluded)}):")
                for v in excluded:
                    print(f"   {v['id']}  {v['title'][:55]}")

            if done:
                print(f"\n\u2705 Already downloaded ({len(done)}):")
                for v in done[:10]:
                    print(f"   {v['id']}  {v['title'][:55]}")
                if len(done) > 10:
                    print(f"   ... +{len(done) - 10} more")

    btn_save.on_click(_on_save)
    btn_fetch.on_click(_on_fetch)

    # ── Layout ─────────────────────────────────────────────
    display(widgets.HTML("<h3>\U0001f50d Browse & Filter</h3>"))
    display(dd_ch)
    display(widgets.HTML(
        "<b>Duration filter</b> <small>(minutes, 0 = no limit)</small>"
    ))
    display(widgets.HBox([int_min, int_max]))
    display(widgets.HTML(
        "<b>Title filter</b> <small>(regex, case-insensitive)</small>"
    ))
    display(widgets.HBox([txt_inc, txt_exc]))
    display(widgets.HTML(
        "<b>Manual exclusions</b> <small>(video IDs, one per line)</small>"
    ))
    display(txt_ids)
    display(widgets.HBox([btn_save, btn_fetch]))
    display(out_pv)

In [ ]:
#@title 3) Sync — Download filtered videos
#@markdown Downloads are pipelined: next video downloads while previous copies to Drive.
#@markdown Stops immediately on cookie/auth failure.

from concurrent.futures import ThreadPoolExecutor, Future

out_sync = widgets.Output()
btn_sync = widgets.Button(
    description="\u25b6 Start Sync",
    button_style="success",
    layout=widgets.Layout(width="200px", height="40px"),
)

def _on_sync(_):
    btn_sync.disabled = True
    with out_sync:
        clear_output()
        chs = config.get("channels", [])
        if not chs:
            print("No channels configured.")
            btn_sync.disabled = False
            return

        total_ok, total_fail = 0, 0
        executor = ThreadPoolExecutor(max_workers=1)

        try:
            for ch in chs:
                cid = ch.get("channel_id", "")
                inp = ch["input"]
                print(f"\n{'=' * 60}")
                print(f"\U0001f4fa {inp}  ({cid})")
                print(f"{'=' * 60}")

                # Fetch or use cache
                if inp in _video_cache:
                    videos = _video_cache[inp]
                    print(f"   Cached: {len(videos)} videos")
                else:
                    print(f"   Fetching...")
                    try:
                        cid, videos = fetch_channel_videos(inp)
                        _video_cache[inp] = videos
                    except CookieExpiredError as e:
                        print(f"\U0001f6d1 {e}")
                        raise
                    except Exception as e:
                        print(f"   \u274c {e}")
                        continue

                passing, _, _ = apply_filters(videos, ch.get("filters", {}))
                cdir = Path(LENS_COMPRESSION_YOUTUBE_ROOT) / inp
                cdir.mkdir(parents=True, exist_ok=True)

                mp = cdir / "_channel.json"
                if not mp.exists():
                    mp.write_text(json.dumps({
                        "input": inp, "channel_id": cid,
                        "added": time.strftime(
                            "%Y-%m-%dT%H:%M:%SZ", time.gmtime()
                        ),
                    }, indent=2))

                already = get_downloaded_ids(cdir)
                to_dl = [v for v in passing if v["id"] not in already]

                print(
                    f"   Filter pass: {len(passing)} \u2502 "
                    f"Done: {len(already)} \u2502 "
                    f"New: {len(to_dl)}"
                )

                if not to_dl:
                    print(f"   \u2705 Up to date!")
                    continue

                pending = None
                for i, v in enumerate(to_dl, 1):
                    vid, title = v["id"], v["title"][:50]
                    print(f"   [{i}/{len(to_dl)}] {vid} \u2014 {title}")

                    dest = cdir / vid
                    if (dest / "audio.wav").exists():
                        print(f"      \u23ed exists")
                        continue

                    if pending:
                        pending.result()

                    print(f"      \u2b07 Downloading...")
                    try:
                        wav = download_to_tmp(vid)
                    except CookieExpiredError as e:
                        print(f"\n\U0001f6d1 {e}")
                        raise

                    if not wav:
                        total_fail += 1
                        continue

                    print(f"      \U0001f4be \u2192 Drive")
                    pending = executor.submit(copy_to_drive, wav, dest)
                    total_ok += 1

                if pending:
                    pending.result()

        except CookieExpiredError:
            executor.shutdown(wait=True)
            print(
                f"\n\U0001f6d1 Stopped \u2014 update cookies.txt at "
                f"{COOKIES_PATH}"
            )
        finally:
            executor.shutdown(wait=True)

        print(f"\n{'=' * 60}")
        print(
            f"\U0001f3c1 Done: {total_ok} downloaded, "
            f"{total_fail} failed"
        )

    btn_sync.disabled = False

btn_sync.on_click(_on_sync)

display(widgets.HTML("<h3>\u2b07 Sync Downloads</h3>"))
display(widgets.HTML(
    "<small>Downloads all filtered videos across all channels. "
    "Pipelined: downloads next while copying previous to Drive.</small>"
))
display(btn_sync)
display(out_sync)